In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error
import joblib

# 1. ЗАГРУЗКА ДАННЫХ
df_train_raw = pd.read_csv("conveyor_dataset_ml.csv")
df_test_raw = pd.read_csv("conveyor_dataset_inference.csv")

def engineer_features_conveyor(data):
    df = data.copy()
    
    # А. Восстановление площади поверхности ленты (если отсутствует)
    if 'belt_surface_area' not in df.columns:
        df['belt_surface_area'] = df['length_ft'] * df['width_in']
    
    # Б. Заполнение пропусков в производительности (flow_tph)
    df['flow_tph'] = df['flow_tph'].fillna(df['flow_tph'].median())
    
    # В. Новые физические признаки
    # Удельная нагрузка (прокси для оценки мощности и усиления рамы)
    df['load_proxy'] = df['flow_tph'] / (df['width_in'] + 1)
    
    # Г. Логарифмирование признаков с широким диапазоном
    cols_to_log = ['length_ft', 'width_in', 'flow_tph', 'belt_surface_area', 'load_proxy']
    for col in cols_to_log:
        df[f'log_{col}'] = np.log1p(df[col])
    
    # Д. Подготовка целевой переменной (в логарифмическом виде)
    if 'weight_kg' in df.columns:
        df['weight_kg_log'] = np.log1p(df['weight_kg'])
        
    return df

# 2. ПОДГОТОВКА ВЫБОРОК
train_df = engineer_features_conveyor(df_train_raw)
test_df = engineer_features_conveyor(df_test_raw)

# Выбираем только логарифмированные признаки
features = [
    'log_length_ft', 'log_width_in', 'log_flow_tph', 
    'log_belt_surface_area', 'log_load_proxy'
]

X_train = train_df[features]
y_train = train_df["weight_kg_log"]
X_test = test_df[features]
y_test = test_df["weight_kg_log"]

# 3. ОБУЧЕНИЕ МОДЕЛИ
base_regressor = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.05,
    max_depth=5,
    l2_regularization=1.0,
    random_state=42
)

# Обертка для автоматической работы с логами (не требует ручного np.exp)
model = TransformedTargetRegressor(
    regressor=base_regressor,
    func=None,         # Цель уже логарифмирована в датасете
    inverse_func=None  
)

model.fit(X_train, y_train)

# 4. ОЦЕНКА КАЧЕСТВА
def evaluate_metrics(model, X, y_log, label=""):
    preds_log = model.predict(X)
    y_true = np.expm1(y_log) # Обратное преобразование из логарифма
    y_pred = np.expm1(preds_log)
    
    print(f"\n=== {label} ===")
    print(f"R²:    {r2_score(y_true, y_pred):.4f}")
    print(f"MAPE:  {mean_absolute_percentage_error(y_true, y_pred)*100:.2f}%")
    print(f"RMSE:  {np.sqrt(mean_squared_error(y_true, y_pred)):.1f} кг")

evaluate_metrics(model, X_test, y_test, "ТЕСТ (CONVEYOR INFERENCE)")

# 5. СОХРАНЕНИЕ
joblib.dump(model, "conveyor_model_final.joblib")


=== ТЕСТ (CONVEYOR INFERENCE) ===
R²:    0.9995
MAPE:  1.65%
RMSE:  77.9 кг


['conveyor_model_final.joblib']